# Agent live observability: deploy infrastructure

This notebook deploys the observability infrastructure at **subscription scope**, fanning out to two resource groups:

- **Spoke RG** (`rg-foundry-multi-{suffix}`) - Log Analytics, Application Insights, `obs-project`, AppInsights connection on the shared spoke account, APIM connection.
- **Admin/core RG** (`rg-foundry-core-{suffix}`) - AppInsights connection on `aif-core-{suffix}` (so the Foundry Portal Monitor tab for admin agents like `aria-rm-briefing-agent` has a backing App Insights), plus the `Foundry User` role on the admin project's managed identity.

## What gets deployed

| Resource | Where | Purpose |
|----------|-------|---------|
| `log-obs-{suffix}` | Spoke RG | Log Analytics Workspace (App Insights backend) |
| `appi-obs-{suffix}` | Spoke RG | Application Insights - stores and visualizes agent traces |
| `obs-project` | Spoke account | Foundry Project on the existing shared account |
| `appinsights-connection` | Spoke account | Account-scoped AppInsights connection (`isSharedToAll: true`) |
| `landing-zone-apim` | `obs-project` | Project-scoped APIM connection for model access via gateway |
| `appinsights-connection` | **Core account** | Same App Insights resource attached to `aif-core` so Aria's Portal Monitor tab works |
| `Foundry User` role | **Admin project MI** | Required by Portal Monitor tab + continuous-eval rules |

## Prerequisites

1. **Python environment**: Run `uv sync` from the repository root to create the shared `.venv`, then select the `.venv` kernel in VS Code.
2. **Previous project setup complete** - both the multi (spoke) and core (hub) Foundry estates must be deployed.
3. **Azure CLI** authenticated - run `az login` before executing cells.
4. **Permissions** - Owner or Contributor + User Access Administrator on both `rg-foundry-multi-{suffix}` and `rg-foundry-core-{suffix}`.

## References

- [Agent Tracing Overview](https://learn.microsoft.com/azure/ai-foundry/observability/concepts/trace-agent-concept?view=foundry)
- [Set Up Tracing in Microsoft Foundry](https://learn.microsoft.com/azure/ai-foundry/observability/how-to/trace-agent-setup?view=foundry)

In [1]:
import os, subprocess, hashlib
from pathlib import Path
from dotenv import load_dotenv

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

SUB_ID   = subprocess.run('az account show --query id -o tsv', shell=True, capture_output=True, text=True).stdout.strip()
SUFFIX   = hashlib.sha256((SUB_ID + 'v2').encode()).hexdigest()[:6]

GATEWAY_URL            = os.environ['GATEWAY_URL']
MULTI_ACCOUNT          = os.environ['MULTI_ACCOUNT']
MULTI_ACCOUNT_ENDPOINT = os.environ['MULTI_ACCOUNT_ENDPOINT']
CHAT_MODEL             = os.environ['CHAT_MODEL']

# Spoke RG hosts the App Insights / Log Analytics / obs-project / spoke account.
# Core RG hosts the admin account (aif-core-{suffix}) where Aria lives - we
# need a Foundry-account-level App Insights connection there too so the Portal
# Monitor tab for admin agents has an App Insights backing, plus the Azure AI
# User role on the admin project's MI.
RESOURCE_GROUP = f'rg-foundry-multi-{SUFFIX}'
CORE_RG        = f'rg-foundry-core-{SUFFIX}'
CORE_ACCOUNT   = f'aif-core-{SUFFIX}'
ADMIN_PROJECT  = f'project-admin-{SUFFIX}'

# Derive APIM service name from GATEWAY_URL (https://{apim-name}.azure-api.net/openai)
APIM_NAME = GATEWAY_URL.split('//')[1].split('.')[0]

# Pull location from the spoke RG - the new resources (App Insights, Log
# Analytics) deploy there. The admin module only references existing resources,
# so its location is inherited implicitly.
LOCATION = subprocess.run(
    f'az group show -n {RESOURCE_GROUP} --query location -o tsv',
    shell=True, capture_output=True, text=True
).stdout.strip()

print(f'Suffix:         {SUFFIX}')
print(f'Spoke RG:       {RESOURCE_GROUP}')
print(f'Core RG:        {CORE_RG}')
print(f'MULTI_ACCOUNT:  {MULTI_ACCOUNT}')
print(f'CORE_ACCOUNT:   {CORE_ACCOUNT}')
print(f'ADMIN_PROJECT:  {ADMIN_PROJECT}')
print(f'APIM Name:      {APIM_NAME}')
print(f'Location:       {LOCATION}')
print(f'Chat Model:     {CHAT_MODEL}')

Suffix:         c2676f
Spoke RG:       rg-foundry-multi-c2676f
Core RG:        rg-foundry-core-c2676f
MULTI_ACCOUNT:  aif-spoke-multi-c2676f
CORE_ACCOUNT:   aif-core-c2676f
ADMIN_PROJECT:  project-admin-c2676f
APIM Name:      apim-foundry-c2676f
Location:       eastus2
Chat Model:     gpt-4.1-mini


In [2]:
import json, base64

# Get principal ID from JWT token (avoids Graph API network call)
token   = subprocess.run('az account get-access-token --query accessToken -o tsv', shell=True, capture_output=True, text=True).stdout.strip()
payload = token.split('.')[1] + '=='
DEPLOYER_PRINCIPAL_ID = json.loads(base64.b64decode(payload))['oid']

print(f'Principal ID: {DEPLOYER_PRINCIPAL_ID}')

Principal ID: 5db0aa5d-f281-47a3-9720-04727dec61e8


In [3]:
# Create APIM subscription foundry-gateway-obs and retrieve the primary key.
# CORE_RG is set in the config cell above.
sub_name  = 'foundry-gateway-obs'
apim_base = (
    f'https://management.azure.com/subscriptions/{SUB_ID}'
    f'/resourceGroups/{CORE_RG}/providers/Microsoft.ApiManagement/service/{APIM_NAME}'
)

create_result = subprocess.run(
    [
        'az', 'rest', '--method', 'PUT',
        '--uri', f'{apim_base}/subscriptions/{sub_name}?api-version=2024-06-01-preview',
        '--body', json.dumps({
            'properties': {
                'displayName': 'Agent Observability Gateway Access',
                'scope': (
                    f'/subscriptions/{SUB_ID}/resourceGroups/{CORE_RG}'
                    f'/providers/Microsoft.ApiManagement/service/{APIM_NAME}/apis/openai'
                ),
                'state': 'active'
            }
        }),
        '--headers', 'Content-Type=application/json',
    ],
    capture_output=True, text=True
)
if create_result.returncode != 0:
    raise RuntimeError(f'APIM subscription create failed: {create_result.stderr.strip()}')
print(f'APIM subscription created/updated: {sub_name}')

key_result = subprocess.run(
    f'az rest --method POST'
    f' --uri "{apim_base}/subscriptions/{sub_name}/listSecrets?api-version=2024-06-01-preview"'
    f' --query primaryKey -o tsv',
    shell=True, capture_output=True, text=True
)
if key_result.returncode != 0:
    raise RuntimeError(f'listSecrets failed: {key_result.stderr.strip()}')
OBS_GATEWAY_KEY = key_result.stdout.strip()

print(f'OBS_GATEWAY_KEY retrieved: {OBS_GATEWAY_KEY[:2]}... (hidden)')

APIM subscription created/updated: foundry-gateway-obs
OBS_GATEWAY_KEY retrieved: 4d... (hidden)


In [7]:
# Deploy main.bicep at SUBSCRIPTION scope. The template fans out to two
# modules - one in the spoke RG (App Insights, Log Analytics, obs-project,
# spoke account connection, APIM connection) and one in the core RG (Foundry
# account-level App Insights connection on aif-core, plus Foundry User role
# on the admin project's MI).
print(f'Deploying to subscription (spoke RG: {RESOURCE_GROUP}, core RG: {CORE_RG}) - ~2-3 minutes...')

deploy_result = subprocess.run(
    [
        'az', 'deployment', 'sub', 'create',
        '--location', LOCATION,
        '--template-file', 'main.bicep',
        '-p', f'location={LOCATION}',
        '-p', f'spokeResourceGroup={RESOURCE_GROUP}',
        '-p', f'adminResourceGroup={CORE_RG}',
        '-p', f'multiAccountName={MULTI_ACCOUNT}',
        '-p', f'coreAccountName={CORE_ACCOUNT}',
        '-p', f'adminProjectName={ADMIN_PROJECT}',
        '-p', f'deployerPrincipalId={DEPLOYER_PRINCIPAL_ID}',
        '-p', f'apimSubscriptionKey={OBS_GATEWAY_KEY}',
        '-p', f'apimName={APIM_NAME}',
        '--query', 'properties.outputs',
        '-o', 'json',
    ],
    capture_output=True, text=True
)
if deploy_result.returncode != 0:
    raise RuntimeError(f'Deployment failed:\n{deploy_result.stderr}')

outputs = json.loads(deploy_result.stdout)

OBS_PROJECT_ENDPOINT         = outputs['projectEndpoint']['value']
OBS_APIM_CONNECTION          = outputs['apimConnectionName']['value']
OBS_APP_INSIGHTS_NAME        = outputs['appInsightsName']['value']
OBS_APP_INSIGHTS_CONN_STRING = outputs['appInsightsConnectionString']['value']
ADMIN_APPINSIGHTS_CONNECTION = outputs['adminAppInsightsConnectionName']['value']

print('Deployment complete!')
print(f'  obs-project endpoint:           {OBS_PROJECT_ENDPOINT}')
print(f'  APIM connection (spoke):        {OBS_APIM_CONNECTION}')
print(f'  App Insights name:              {OBS_APP_INSIGHTS_NAME}')
print(f'  Admin App Insights connection:  {ADMIN_APPINSIGHTS_CONNECTION} (on {CORE_ACCOUNT})')

Deploying to subscription (spoke RG: rg-foundry-multi-c2676f, core RG: rg-foundry-core-c2676f) — ~2-3 minutes...
Deployment complete!
  obs-project endpoint:           https://aif-spoke-multi-c2676f.services.ai.azure.com/api/projects/obs-project
  APIM connection (spoke):        landing-zone-apim
  App Insights name:              appi-obs-n5d3ja
  Admin App Insights connection:  appinsights-connection (on aif-core-c2676f)


## Patch APIM connection credentials

The Bicep deployment creates the `landing-zone-apim` connection on `obs-project`, but the `OBS_GATEWAY_KEY` is a sensitive value retrieved at runtime and passed as a `@secure()` parameter - it is intentionally excluded from deployment history. This PATCH call ensures the live credential on the connection matches the key retrieved above.

It is safe to re-run: if the key is already correct the PATCH is a no-op.

In [5]:
import tempfile

# PATCH the landing-zone-apim connection on obs-project with the OBS_GATEWAY_KEY
models_list = [{"name": CHAT_MODEL, "properties": {"model": {"name": CHAT_MODEL, "version": "", "format": "OpenAI"}}}]
connection_uri = (
    f'https://management.azure.com/subscriptions/{SUB_ID}'
    f'/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.CognitiveServices/accounts/{MULTI_ACCOUNT}'
    f'/projects/obs-project/connections/{OBS_APIM_CONNECTION}?api-version=2025-04-01-preview'
)

with tempfile.NamedTemporaryFile(mode='w', suffix='.json', delete=False) as f:
    json.dump({
        'properties': {
            'category': 'ApiManagement',
            'target': GATEWAY_URL,
            'authType': 'ApiKey',
            'credentials': {'key': OBS_GATEWAY_KEY},
            'metadata': {
                'deploymentInPath': 'true',
                'inferenceAPIVersion': '2024-10-21',
                'models': json.dumps(models_list),
            },
        }
    }, f)
    payload_file = f.name

patch_result = subprocess.run(
    f'az rest --method PATCH --uri "{connection_uri}"'
    f' --body @"{payload_file}" --headers "Content-Type=application/json" -o json',
    shell=True, capture_output=True, text=True
)
if patch_result.returncode != 0:
    print(f'PATCH warning (connection may already be correct): {patch_result.stderr.strip()}')
else:
    print(f'Connection patched: {OBS_APIM_CONNECTION} on obs-project')

Connection patched: landing-zone-apim on obs-project


In [6]:
# Write OBS_* keys to repo root .env using read-modify-write pattern
env_file = repo_root / '.env'
existing = {}
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if '=' in line and not line.startswith('#'):
            k, _, v = line.partition('=')
            existing[k.strip()] = v.strip()

existing.update({
    'OBS_FOUNDRY_PROJECT_ENDPOINT': OBS_PROJECT_ENDPOINT,
    'OBS_APIM_CONNECTION':          OBS_APIM_CONNECTION,
    'OBS_GATEWAY_KEY':              OBS_GATEWAY_KEY,
    'OBS_APP_INSIGHTS_NAME':        OBS_APP_INSIGHTS_NAME,
    'OBS_APP_INSIGHTS_CONN_STRING': OBS_APP_INSIGHTS_CONN_STRING,
    'OBS_RESOURCE_GROUP':           RESOURCE_GROUP,
})

env_file.write_text('\n'.join(f'{k}={v}' for k, v in existing.items()) + '\n')
print('.env updated with OBS_* keys:')
for key in ['OBS_FOUNDRY_PROJECT_ENDPOINT', 'OBS_APIM_CONNECTION', 'OBS_GATEWAY_KEY',
            'OBS_APP_INSIGHTS_NAME', 'OBS_APP_INSIGHTS_CONN_STRING', 'OBS_RESOURCE_GROUP']:
    print(f'  {key}')

.env updated with OBS_* keys:
  OBS_FOUNDRY_PROJECT_ENDPOINT
  OBS_APIM_CONNECTION
  OBS_GATEWAY_KEY
  OBS_APP_INSIGHTS_NAME
  OBS_APP_INSIGHTS_CONN_STRING
  OBS_RESOURCE_GROUP


## Infrastructure summary

The following `.env` keys have been written. Run [`08-07-03-agent-observability.ipynb`](08-07-03-agent-observability.ipynb) to trace the Aria agent and verify traces in Application Insights.

| `.env` key | Purpose |
|------------|---------|
| `OBS_FOUNDRY_PROJECT_ENDPOINT` | `obs-project` endpoint on the spoke account (legacy - not used by the Aria-pivoted notebooks; preserved for backward compatibility) |
| `OBS_APIM_CONNECTION` | APIM connection name (`landing-zone-apim`) on `obs-project` |
| `OBS_GATEWAY_KEY` | APIM subscription key (`foundry-gateway-obs`) |
| `OBS_APP_INSIGHTS_NAME` | Application Insights resource name (`appi-obs-{suffix}`) |
| `OBS_APP_INSIGHTS_CONN_STRING` | App Insights connection string for `configure_azure_monitor()` |
| `OBS_RESOURCE_GROUP` | Resource group (`rg-foundry-multi-{suffix}`) - used for KQL resource ID construction |

The admin-account App Insights connection is also wired up by this deployment (see the deployment output above) - no `.env` key needed because the active notebooks ([08-07-03](08-07-03-agent-observability.ipynb), [08-07-05](08-07-05-agent-continuous-evaluation.ipynb)) derive the admin endpoint deterministically from `SUFFIX`.